In [1]:
import torch
print(torch.__version__)
x = torch.tensor([1.0, 2.0, 3.0])
print(x)

2.10.0+cpu
tensor([1., 2., 3.])


In [2]:
x = torch.tensor([1.0, 2.0, 3.0])
y_true = torch.tensor([2.0, 4.0, 6.0])

print("Input:", x)
print("Target:", y_true)

Input: tensor([1., 2., 3.])
Target: tensor([2., 4., 6.])


In [3]:
w = torch.tensor(0.0, requires_grad=True)
print("Starting weight:", w)

Starting weight: tensor(0., requires_grad=True)


In [4]:
def forward(x):
    return w * x

y_pred = forward(x)
print("Predictions (before training):", y_pred)

Predictions (before training): tensor([0., 0., 0.], grad_fn=<MulBackward0>)


In [5]:
def loss_fn(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean()

loss = loss_fn(y_pred, y_true)
print("Initial loss:", loss.item())

Initial loss: 18.66666603088379


In [6]:
learning_rate = 0.01

for epoch in range(50):

    # Forward pass
    y_pred = forward(x)

    # Compute loss
    loss = loss_fn(y_pred, y_true)

    # Backward pass - compute gradients
    loss.backward()

    # Update weight
    with torch.no_grad():
        w -= learning_rate * w.grad

    # Reset gradients
    w.grad.zero_()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | w: {w.item():.4f}")

Epoch   0 | Loss: 18.6667 | w: 0.1867
Epoch  10 | Loss: 2.6304 | w: 1.3193
Epoch  20 | Loss: 0.3707 | w: 1.7445
Epoch  30 | Loss: 0.0522 | w: 1.9041
Epoch  40 | Loss: 0.0074 | w: 1.9640


In [7]:
test_input = torch.tensor([4.0])
prediction = forward(test_input)
print(f"Input: 4.0 → Prediction: {prediction.item():.4f} (should be close to 8.0)")

Input: 4.0 → Prediction: 7.9404 (should be close to 8.0)


#### Experiment 1 - More epochs

In [12]:
w = torch.tensor(0.0, requires_grad=True)  # reset weight

learning_rate = 0.01

for epoch in range(200):  # changed from 50 to 200
    y_pred = forward(x)
    loss = loss_fn(y_pred, y_true)
    loss.backward()
    with torch.no_grad():
        w -= learning_rate * w.grad  # This is gradient descent. The formula is:
# w = w - (learning_rate × gradient)
    w.grad.zero_()

if True:
    print(f"Final Loss: {loss.item():.6f} | w: {w.item():.6f}")
    prediction = forward(torch.tensor([4.0]))
    print(f"Prediction: {prediction.item():.4f}") 
# The full picture in one sentence per step: Forward-Model guesses , Loss-You measure how wrong , BackwardPyTorch computes how to improve , UpdateYou move w in the right direction , Zero gradYou clear the slate for next round

Final Loss: 0.000000 | w: 1.999999
Prediction: 8.0000


### Line 1 — Reset the weight
pythonw = torch.tensor(0.0, requires_grad=True)
You start fresh. Weight is 0 — the model knows nothing.
requires_grad=True tells PyTorch: track everything that happens to this value, because I will need to learn from it.

### Line 2 — Set learning rate
pythonlearning_rate = 0.01
This controls how big each update step is. Too large → the model overshoots and never converges. Too small → learning is painfully slow. 0.01 is a safe starting point.

### Line 3 — Start the loop
pythonfor epoch in range(200):
One epoch = one full pass through your data. You repeat this 200 times, each time making the model slightly better.

### Line 4 — Forward pass
pythony_pred = forward(x)
The model makes its current best guess. Early on this is terrible. By epoch 200 it's nearly perfect. Internally this runs w * x.

### Line 5 — Compute loss
pythonloss = loss_fn(y_pred, y_true)
```
Measures how wrong the prediction is. Specifically it computes:
```
mean( (prediction - truth)² )
Large number = very wrong. Zero = perfect.

### Line 6 — Backward pass
pythonloss.backward()
This is where learning happens. PyTorch looks at the loss and works backwards through every operation to answer one question:
"If I increase w slightly, does the loss go up or down — and by how much?"
That answer is stored in w.grad.

### Line 7 — Update the weight
pythonwith torch.no_grad():
    w -= learning_rate * w.grad
```
This is gradient descent. The formula is:
```
w = w - (learning_rate × gradient)
torch.no_grad() wraps it because this is a manual update — you don't want PyTorch tracking this step as part of the learning graph.
If the gradient is positive → w was too high → subtract to bring it down.
If the gradient is negative → w was too low → subtracting a negative adds to it.
Either way, w moves toward the correct value.

### Line 8 — Reset gradients
pythonw.grad.zero_()
PyTorch accumulates gradients by default — it adds each new gradient on top of the old one. If you don't reset, by epoch 10 your gradient is 10x too large and the model explodes. You must zero it every single loop.

#### The print block
pythonprint(f"Final Loss: {loss.item():.6f} | w: {w.item():.6f}")
prediction = forward(torch.tensor([4.0]))
print(f"Prediction: {prediction.item():.4f}")
.item() pulls the number out of the tensor so you can read it cleanly. The format :.6f means 6 decimal places.

####  This exact cycle — forward, loss, backward, update, zero — is what happens inside every neural network that has ever been trained